Distribuciones de probabilidad correctas para describir a las variables.

In [6]:
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import numpy as np

# Cargar datos
df = pd.read_csv("Effor_index_MASTER.csv")

# Columna de existencia de costo
df["costo"] = df["TraCosto"].map({"VERDADERO": 1, "FALSO": 0})

mapa_fijo = {
    "nivel 1": 1,
    "nivel 2": 2,
    "nivel 3.1": 3,
    "nivel 3.2": 4,
    "nivel 3.3": 5,
    "nivel 3.4": 6,
    "nivel 3.5": 7,
    "nivel 3.6": 8,
    "nivel 3.7": 9,
    "nivel 3.8": 10,
    "nivel 3.9": 11,
    "nivel 4.1": 12,
    "nivel 4.2": 13,
    "nivel 4.3": 14
}

df["nivel_digitalizacion_numT"] = (
    df["nivel_digitalizacion"]
    .str.lower()
    .str.strip()
    .map(mapa_fijo)
)

df["nivel_digitalizacion_num"] = df["nivel_digitalizacion_numT"].fillna(0)

Las variables presentes en trámites y servicios son de cierta categoría. El nivel de digitalización es discreto habiendo solo 14 niveles; los números de formatos y requisitos también son discretos obteniendo un valor dependiendo del trámite; el costo es un valor binario, a fin de cuentas discreto, que determina si existe costo o no del trámite; finalmente, el tiempo, al ser una variable de valores muy grandes, su rango existe entre los 0 y los 518 mil minutos, prácticamente este dato permite cualquier valor en ese intervalo, por ello es posible considerarla como continua.

Para la propuesta de un índice de esfuerzo, se propone el uso de la entropía de Shannon, un valor que determina incertidumbre o dispersión en los datos. Este método utiliza variables discretas y las probabilidades de que exista dicha variable entre el rango de posibles valores, por ello, será necesario poder discretizar el valor del tiempo.

Como los datos de tiempo caen dentro del rango de 0 a 518 mil minutos (1 año aproximadamente) podemos proponer el reescalamiento de dicha variable en un rango más corto utilizando el logaritmo de los valores de tiempo. Como existen tiempos cero entonces sería necesario incluir el término "1+t" para el logaritmo. Podemos asignar el rango de la transformación en deciles.

In [7]:
# Transformación log(1 + t)
df["tiempo_log"] = np.log1p(df["Tiempo_en_minutos"])

df["tiempo_decil"] = pd.qcut(df["tiempo_log"], 10, labels=False)

df["tiempo_decil"].value_counts().sort_index()

bins = pd.qcut(df["tiempo_log"], 10)
print(bins.cat.categories)

intervals = bins.cat.categories

for i, interval in enumerate(intervals):
    low = np.expm1(interval.left)
    high = np.expm1(interval.right)
    print(f"Decil {i}: {low:.1f} - {high:.1f} minutos")

IntervalIndex([ (-0.001, 2.773],   (2.773, 3.317],   (3.317, 3.434],
                 (3.434, 5.504],   (5.504, 7.273],   (7.273, 8.371],
                 (8.371, 9.064],    (9.064, 9.98],   (9.98, 10.674],
               (10.674, 13.159]],
              dtype='interval[float64, right]')
Decil 0: -0.0 - 15.0 minutos
Decil 1: 15.0 - 26.6 minutos
Decil 2: 26.6 - 30.0 minutos
Decil 3: 30.0 - 244.7 minutos
Decil 4: 244.7 - 1439.9 minutos
Decil 5: 1439.9 - 4319.0 minutos
Decil 6: 4319.0 - 8637.6 minutos
Decil 7: 8637.6 - 21589.3 minutos
Decil 8: 21589.3 - 43216.5 minutos
Decil 9: 43216.5 - 518657.0 minutos


Ahora determinamos la entropía de cada una de las variables

In [8]:
#Probabilidad de los valores
probs = df["tiempo_decil"].value_counts(normalize=True)

#Entropía
H = -np.sum(probs * np.log(probs))
print(H)

2.2758908094307526


In [10]:
df["N_FORMATOS_FINAL"].value_counts().sort_index()

N_FORMATOS_FINAL
0     305
1     296
2      45
3      13
4       4
5       1
6       1
12      1
Name: count, dtype: int64